# EDA

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('../../DATABASE/Samples_vs_Genes_new_29Sep_s.csv', index_col='Unnamed: 0')
label = pd.read_csv('../../DATABASE/COE_COPPER_dataset_response_sn.csv',index_col='id')

/var/folders/b7/2l_r5k0s0hd43f58p5pr03qw0000gn/T/ipykernel_23692/3607881980.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../../DATABASE/Samples_vs_Genes_new_29Sep_s.csv', index_col='Unnamed: 0')


In [3]:
y = label.loc[data.index.astype(str)].response.to_numpy().reshape(-1,1)
# y = np.load('sydney_data_label.npy')
# x = np.load('sydney_data.npy')

In [4]:
print(f'number of samples: {data.shape[0]}')
print(f'number of genes:   {data.shape[1]}')

number of samples: 30558
number of genes:   16383


In [6]:
data

,CATG00000000247,CATG00000000273,CATG00000000280,CATG00000000282,CATG00000000300,CATG00000000313,CATG00000000321,CATG00000000569,CATG00000000711,CATG00000000720,...,ENSG00000227294,ENSG00000227306,ENSG00000227307,ENSG00000227321,ENSG00000227330,ENSG00000227336,ENSG00000227338,ENSG00000227342,ENSG00000227344,ENSG00000227345
1002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1013,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1016,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1018,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1019,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
yin_16_ASD_discovery_controls-control7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
yin_16_ASD_discovery_controls-control8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
yin_16_ASD_discovery_controls-control9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
yin_16_ASD_discovery_controls-controlREF100653,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# this is imbalance data
label.response.value_counts()

response
1    24105
0     6453
Name: count, dtype: int64

In [ ]:
# minimum value for each sample
data.min(axis=1).value_counts()

In [ ]:
# maximum value for each sample
data.max(axis=1).value_counts()

In [ ]:
# describe sum of gene values for each sample
data.sum(axis=1).describe()

In [ ]:
# number of samples with zero value for all genes
sum((data != 0).sum(axis=1) == 0)

In [ ]:
# number of genes with zero values for all sample 
sum((data != 0).sum(axis=0) == 0)

# Model

In [ ]:
y = label.loc[data.index.astype(str)].response.to_numpy().reshape(-1,1)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x = np.load('sydney_data.npy')
y = np.load('sydney_data_label.npy')

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.05, stratify=y, random_state=100)

In [ ]:
print('x_train: ', x_train.shape)
print('y_train: ', y_train.shape)
print('x_test:  ', x_test.shape)
print('y_test:  ', y_test.shape)

In [ ]:
from network.network_builder import get_tissue_specific_maps
import pandas as pd
import importlib
from model.model_utils import plot_history

In [ ]:
Network_params_file = 'Configs/Network_params.py'
loader = importlib.machinery.SourceFileLoader('Network_params', Network_params_file)
Network_params = loader.load_module()

In [ ]:
# for the second time load it 
import pickle
with open('tissue_maps.pkl', 'rb') as f:
    tissue_maps = pickle.load(f)

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers.legacy import Adam

model = Sequential()
model.add(Dense(100, input_shape=(x.shape[1],), activation='relu'))
model.add(Dense(50, activation='relu'))
model.add(Dense(10, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

optimizer = Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
hist = model.fit(x=x_train,
                 y=y_train,
                 validation_data=(x_test, y_test),
                 epochs=40,
                 batch_size=250,
                 verbose=2)

In [ ]:
model.evaluate(x_test, y_test)

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
hist.history['loss']

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1,2,1)
plt.plot(hist.history['loss'], label='loss')
plt.plot(hist.history['val_loss'], label='val_loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(hist.history['accuracy'], label='accuracy')
plt.plot(hist.history['val_accuracy'], label='val_acc')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
hist.history

,CATG00000000247,CATG00000000273,CATG00000000280,CATG00000000282,CATG00000000300,CATG00000000313,CATG00000000321,CATG00000000569,CATG00000000711,CATG00000000720,...,ENSG00000227294,ENSG00000227306,ENSG00000227307,ENSG00000227321,ENSG00000227330,ENSG00000227336,ENSG00000227338,ENSG00000227342,ENSG00000227344,ENSG00000227345
1002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1013,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1016,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1018,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1019,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
yin_16_ASD_discovery_controls-control7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
yin_16_ASD_discovery_controls-control8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
yin_16_ASD_discovery_controls-control9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
yin_16_ASD_discovery_controls-controlREF100653,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
